# ShopDesk, Module 3 Section 1 Lab 2: Path-Scoped Rules and Conditional Loading

A beginner-friendly notebook on `.claude/rules/` with a **paths** field: rules that load **only** when
Claude touches a matching file, so instructions stay relevant and context stays small. We build test and
API rules, validate that each file triggers the right rules, and compare the token cost against a single
**monolithic** `CLAUDE.md`. Everything runs offline in a sandbox; a live **Claude Agent SDK** cell loads the
project rules. Runs **Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

ShopDesk's test files and API files have different conventions. Putting both sets of rules in one
`CLAUDE.md` means every session pays for every rule, even when editing an unrelated file. Path-scoped rules
fix this: the testing rule loads only for test files, the API rule only for API files, and a shared style
rule loads for everything.

The question this lab answers: **how do path globs decide which rules load, and how much context does that
save versus one big file?**

## Objectives

- Write `.claude/rules/` files with a **paths** field of glob patterns.
- Validate that each file loads exactly the rules whose patterns match (plus any unconditional rule).
- Compare the token cost of path-scoped rules against a **monolithic** `CLAUDE.md`.

## What you'll observe

- A test file triggers the testing rule; an API file triggers the API rule; neither triggers the other.
- A rule with no `paths` loads for every file.
- Across a set of files, the modular setup loads fewer tokens than the monolithic file.

## How to run

Run top to bottom. Building the sandbox, matching globs, and the comparison run anywhere. The live cell
loads the project rules through Claude, so paste a real key into **Setup 2/3** and re-run from the top;
otherwise it skips. **Node.js 18+** is needed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. `pyyaml` parses the rule frontmatter; the Agent SDK drives the
live cell and needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q claude-agent-sdk anthropic python-dotenv pyyaml

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sandbox
import re                                       # glob-to-regex matching
import sys                                       # detect Windows (special event loop)
import yaml                                     # parse the rule frontmatter
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds a sandbox project with three rules and a few source files. `testing.md` scopes
to test files, `api.md` scopes to the API folder, and `style.md` has no `paths` so it applies everywhere.
The source files are what we will match against.

In [ ]:
# ===== SETUP 3/3 - build the sandbox project (rules + files) =====
import textwrap                                    # keeps the embedded file bodies readable
PROJECT = os.path.join(os.getcwd(), "rules_sandbox")   # the sample project root

def write(rel, content):                           # small helper: write a file under PROJECT
    path = os.path.join(PROJECT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

write(".claude/rules/testing.md", textwrap.dedent("""\
    ---
    paths:
      - "**/*.test.tsx"
      - "**/*.test.ts"
    ---
    # Testing rules
    - Name tests "should [action] when [condition]".
    - Mock external services; never call real APIs.
    """))
write(".claude/rules/api.md", textwrap.dedent("""\
    ---
    paths:
      - "src/api/**/*.ts"
    ---
    # API rules
    - Validate every input at the boundary.
    - Return the standard error shape.
    """))
write(".claude/rules/style.md", textwrap.dedent("""\
    # Style rules (no paths -> always loaded)
    - Use 2-space indentation.
    - Prefer clear names over comments.
    """))
for f in ["src/api/orders.ts", "src/components/Button.tsx",
          "src/components/Button.test.tsx", "src/util/math.ts"]:
    write(f, "// " + f + "\n")                      # tiny placeholder source files
print("project at", PROJECT)

### How path scoping works

- Every `.md` in `.claude/rules/` loads as project memory. A rule **without** a `paths` field loads
  **unconditionally**, like `CLAUDE.md`.
- A rule **with** a `paths` field (a YAML list of globs) loads **only** when Claude works on a file matching
  one of those globs, at match time rather than at startup.
- That conditional loading is where the context savings come from; a pile of unscoped rule files costs the
  same as one big file.

One gotcha: user-level `~/.claude/rules/` with `paths` is currently ignored (a known bug), so keep
path-scoped rules at the **project** level.

---

### Lab objective - load the right rules, and only those

**What you build:** a glob matcher, a `rules_for(file)` resolver, a validation against expected results, and
a monolithic-versus-modular token comparison.

**Why it helps you build real solutions:** path scoping is what keeps a large repo's instructions relevant
and cheap instead of a wall of always-on rules.

**How you'll see it:** each file loads exactly its rules, and the modular total is smaller.

**This cell:** a small **glob matcher**. It converts a glob (`**/*.test.tsx`, `src/api/**/*.ts`) into a
regex: `**/` spans any folders, `*` stays within one path segment. We use `re.escape` for literal characters
so the matching is exact.

In [ ]:
# ===== match a path against a glob =====
def glob_to_regex(pat):                            # glob string -> compiled regex
    out = ""; i = 0
    while i < len(pat):
        if pat[i:i+3] == "**/":                     #   ** plus slash spans directories
            out += "(?:.*/)?"; i += 3
        elif pat[i:i+2] == "**":                    #   bare ** spans anything
            out += ".*"; i += 2
        elif pat[i] == "*":                         #   * stays within one segment
            out += "[^/]*"; i += 1
        elif pat[i] == "?":                         #   ? is one non-slash char
            out += "[^/]"; i += 1
        else:                                       #   everything else is literal
            out += re.escape(pat[i]); i += 1
    return re.compile("^" + out + "$")

def glob_match(pattern, path):                     # True if path matches the glob
    return glob_to_regex(pattern).match(path) is not None

print("Button.test.tsx vs **/*.test.tsx :", glob_match("**/*.test.tsx", "src/components/Button.test.tsx"))
print("orders.ts       vs src/api/**/*.ts:", glob_match("src/api/**/*.ts", "src/api/orders.ts"))
print("math.ts         vs src/api/**/*.ts:", glob_match("src/api/**/*.ts", "src/util/math.ts"))

**This cell:** parse a rule file into its **paths** and body. We split the YAML frontmatter between the
`---` fences and read the `paths` list; a rule with no frontmatter has no paths and so loads
unconditionally.

In [ ]:
# ===== parse a rule into (paths, body) =====
def parse_rule(rel):                               # rule path -> (list_of_globs, body_text)
    text = open(os.path.join(PROJECT, rel)).read()
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)   # fenced frontmatter?
    if not m:
        return [], text                             #   no frontmatter -> unconditional rule
    front = yaml.safe_load(m.group(1)) or {}        #   parse the YAML
    return front.get("paths", []), m.group(2)       #   the globs and the body

for rel in [".claude/rules/testing.md", ".claude/rules/api.md", ".claude/rules/style.md"]:
    paths, _ = parse_rule(rel)
    print(f"  {rel.split('/')[-1]:12} paths -> {paths or '(none: always loads)'}")

**This cell:** the resolver: **which rules load for a file**. A rule loads if it has no paths
(unconditional) or if any of its globs match the file. This is the conditional-loading decision Claude Code
makes when it opens a file.

In [ ]:
# ===== which rules load for a given file =====
RULES = [".claude/rules/testing.md", ".claude/rules/api.md", ".claude/rules/style.md"]

def rules_for(file_path):                          # file -> the rule files that load for it
    loaded = []
    for rel in RULES:
        paths, _ = parse_rule(rel)
        if not paths or any(glob_match(p, file_path) for p in paths):   # unconditional OR a glob matches
            loaded.append(rel.split("/")[-1])
    return loaded

for f in ["src/components/Button.test.tsx", "src/api/orders.ts", "src/util/math.ts"]:
    print(f"  {f:32} -> {rules_for(f)}")

**This cell:** **validate** the conditional loading against what we expect. The test file should load
testing plus style; the API file api plus style; the plain file only style. We assert each case so a wrong
glob would fail loudly.

In [ ]:
# ===== validate conditional loading =====
EXPECTED = {
    "src/components/Button.test.tsx": ["testing.md", "style.md"],
    "src/api/orders.ts":              ["api.md", "style.md"],
    "src/util/math.ts":               ["style.md"],
}
all_ok = True
for f, expected in EXPECTED.items():               # check each file against its expected rule set
    got = rules_for(f)
    ok = got == expected
    all_ok = all_ok and ok
    print(f"  {'OK ' if ok else 'BAD'} {f:32} -> {got}")
print("all conditional-loading checks passed:", all_ok)

**This cell:** the **payoff**: modular versus monolithic. The monolithic `CLAUDE.md` holds every rule
body and loads all of it for every file. The modular setup loads only the rules that apply. We approximate
tokens as characters over four and total the cost across all four files.

In [ ]:
# ===== compare token cost: monolithic vs path-scoped =====
def body_tokens(rel):                              # approximate tokens of a rule body
    _, body = parse_rule(rel)
    return len(body) // 4

monolithic_per_file = sum(body_tokens(r) for r in RULES)   # everything, every file
files = ["src/components/Button.test.tsx", "src/api/orders.ts",
         "src/components/Button.tsx", "src/util/math.ts"]

modular_total = 0
for f in files:                                    # modular: only the rules that load for this file
    modular_total += sum(body_tokens(".claude/rules/" + name) for name in rules_for(f))
monolithic_total = monolithic_per_file * len(files)

print("monolithic total tokens:", monolithic_total)
print("path-scoped total tokens:", modular_total)
print(f"path scoping saved {1 - modular_total / monolithic_total:.0%} across {len(files)} files")

**This cell:** the live view. We point the Agent SDK at the project with `cwd=PROJECT` and
`setting_sources=["project"]`, which loads the project rules, then ask it to work on a test file so the
testing rule is in scope. This shows path-scoped rules reaching the model.

In [ ]:
# ===== live: load project rules and work on a test file =====
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ToolUseBlock

RULE_OPTS = ClaudeAgentOptions(                    # load project rules from the sandbox
    model=MODEL, cwd=PROJECT,
    setting_sources=["project"],                    # this is what loads .claude/rules
    allowed_tools=["Read", "Grep", "Glob"])         # read-only exploration

async def ask(prompt):                              # stream tool calls and the answer
    async for m in query(prompt=prompt, options=RULE_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  ->", b.name)
                elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: ask("Open src/components/Button.test.tsx and summarize the testing conventions that apply."))
else:
    print("[skipped] expected: the testing rule is in scope for the .test.tsx file (naming, mocking).")

| anti-pattern | what to do instead |
|---|---|
| one monolithic `CLAUDE.md` for every convention | split into path-scoped `.claude/rules/` |
| many rule files with no `paths` | they all load at launch; add `paths` to scope them |
| directory `CLAUDE.md` for cross-directory conventions | use a path-scoped rule; no nested tree needed |
| path-scope rules in `~/.claude/rules/` | that is currently ignored; keep them project-level |

**Lesson:** a `paths` field turns a rule into a conditional one that loads only when Claude touches a
matching file, keeping instructions relevant and context small. Unconditional rules (no `paths`) still load
for everything, so reach for path scoping when a convention belongs to only part of the codebase, and prefer
it over directory `CLAUDE.md` for cross-directory rules.

---

## Recap - path-scoped rules

| Rule | paths | Loads for |
|---|---|---|
| testing.md | `**/*.test.tsx`, `**/*.test.ts` | test files only |
| api.md | `src/api/**/*.ts` | API files only |
| style.md | (none) | every file |

One principle to carry forward: **scope a rule to the files it is about, and only pay for it there.** To run
live, paste a real key into **Setup 2/3** and re-run from the top. Then try it: add a rule scoped to
`src/components/**/*.tsx` and confirm it loads for the button files but not the API file.